# Biological Interpretation of the Layer-1 MLP Promoter Signal

## Objective

Identify the sequence features associated with the transferable promoter-class representation found in the Layer-1 MLP of baseline DNABERT-2.

## Previous causal result

Layer-1 MLP outputs from promoter sequences strongly increased the fixed Layer-5 promoter-probe score when patched into dinucleotide-shuffled promoter recipients.

Matched and unrelated promoter donors produced similar effects, while non-promoter donors produced no reliable average rescue.

This indicates that the Layer-1 MLP carries a transferable promoter-class representation rather than primarily matched-sequence-specific information.

## Main questions

1. Which sequence features distinguish promoter donors from non-promoter donors?
2. Which features predict Layer-1 MLP rescue magnitude?
3. Is the effect explained by GC content or CpG density?
4. Are known promoter-associated motifs enriched in high-rescue donors?
5. Does biological feature strength explain rescue beyond the donor's final Layer-5 probe score?

## Candidate features

- GC content
- CpG count and density
- dinucleotide frequencies
- TATA-like motifs
- initiator-like motifs
- GC-box-like motifs
- short k-mer frequencies
- sequence complexity
- promoter probe strength

## Interpretation boundary

This notebook tests biological associations with a diagnostic Layer-5 probe and a Layer-1 MLP intervention. Association alone will not establish that a motif is causally encoded. Controlled sequence interventions will be performed in Notebook 6.

In [1]:
!pip install -q --upgrade --force-reinstall \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    huggingface-hub==0.24.7 \
    datasets \
    joblib \
    scikit-learn \
    scipy \
    ushuffle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 12.8 MB/s eta 0:00:00
 

In [ ]:
import os
os.kill(os.getpid(), 9)

 Cell 1 — Verify compatible environment

In [1]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import datasets
import torch
import transformers
import tokenizers
import huggingface_hub
import fsspec
import requests

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("fsspec:", fsspec.__version__)
print("Requests:", requests.__version__)
print("CUDA available:", torch.cuda.is_available())

NumPy: 2.5.1
Pandas: 3.0.5
SciPy: 1.18.0
Scikit-learn: 1.9.0
Datasets: 4.1.1
Torch: 2.11.0+cu128
Transformers: 4.44.2
Tokenizers: 0.19.1
Hugging Face Hub: 0.24.7
fsspec: 2025.9.0
Requests: 2.34.2
CUDA available: True


# Cell 2 — Mount Drive and verify required files

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

BASE_DIR = Path(
    "/content/drive/MyDrive/algoverse/dnabert"
)

OUTPUT_DIR = BASE_DIR / "outputs"

CAUSAL_DIR = (
    OUTPUT_DIR
    / "promoter_causal_interventions"
)

# Locate the saved Layer-5 probe.
probe_candidates = list(
    BASE_DIR.rglob("*.joblib")
) + list(
    BASE_DIR.rglob("*.pkl")
)

print("Possible saved probe files:")
for path in probe_candidates:
    print(" -", path)

required_files = {
    "negative donor pool":
        CAUSAL_DIR / "negative_donor_pool.csv",

    "negative donor patch results":
        CAUSAL_DIR
        / "layer1_mlp_negative_donor_patch_results.csv",

    "unrelated promoter donor results":
        CAUSAL_DIR
        / "layer1_mlp_unrelated_donor_patch_results.csv",

    "matched promoter patch results":
        CAUSAL_DIR
        / "layer1_mlp_patch_results.csv",

    "causal summary":
        CAUSAL_DIR
        / "causal_intervention_summary.csv",
}

print("\nRequired Notebook 4 files:")

all_required_files_exist = True

for name, path in required_files.items():
    exists = path.exists()

    print(
        f"{name}:",
        "FOUND" if exists else "MISSING",
        "\n  ",
        path,
    )

    all_required_files_exist &= exists

print(
    "\nAll required files found:",
    all_required_files_exist,
)

print("\nFiles currently in causal directory:")

for path in sorted(CAUSAL_DIR.glob("*")):
    print(" -", path.name)

Mounted at /content/drive
Possible saved probe files:
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_baseline_pilot/dnabert_layer5_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_01_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_02_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_03_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_04_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_05_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_06_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_07_probe.joblib
 - /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_component_localization/layer_08_probe.j

# Cell 3 — Load Notebook 4 donor and rescue data

In [3]:
PROBE_PATH = (
    OUTPUT_DIR
    / "promoter_baseline_pilot"
    / "dnabert_layer5_probe.joblib"
)

MATCHED_PATCH_PATH = (
    CAUSAL_DIR
    / "layer1_mlp_clean_to_corrupt_patch_results.csv"
)

UNRELATED_PATCH_PATH = (
    CAUSAL_DIR
    / "layer1_mlp_unrelated_donor_patch_results.csv"
)

NEGATIVE_PATCH_PATH = (
    CAUSAL_DIR
    / "layer1_mlp_negative_donor_patch_results.csv"
)

NEGATIVE_POOL_PATH = (
    CAUSAL_DIR
    / "negative_donor_pool.csv"
)

UNRELATED_STRENGTH_PATH = (
    CAUSAL_DIR
    / "layer1_mlp_unrelated_donor_strength_results.csv"
)

BASELINE_SCORES_PATH = (
    CAUSAL_DIR
    / "clean_corrupt_baseline_scores.csv"
)

# Load fixed Layer-5 diagnostic probe.
layer5_probe = joblib.load(PROBE_PATH)

# Load Notebook 4 intervention outputs.
matched_patch_df = pd.read_csv(
    MATCHED_PATCH_PATH
)

unrelated_patch_df = pd.read_csv(
    UNRELATED_PATCH_PATH
)

negative_patch_df = pd.read_csv(
    NEGATIVE_PATCH_PATH
)

negative_pool_df = pd.read_csv(
    NEGATIVE_POOL_PATH
)

unrelated_strength_df = pd.read_csv(
    UNRELATED_STRENGTH_PATH
)

baseline_scores_df = pd.read_csv(
    BASELINE_SCORES_PATH
)

print("Loaded rows:")
print(" matched promoter patches:", len(matched_patch_df))
print(" unrelated promoter patches:", len(unrelated_patch_df))
print(" non-promoter patches:", len(negative_patch_df))
print(" negative donor pool:", len(negative_pool_df))
print(" unrelated donor strength:", len(unrelated_strength_df))
print(" baseline clean/corrupt scores:", len(baseline_scores_df))

print("\nMatched patch columns:")
print(matched_patch_df.columns.tolist())

print("\nUnrelated patch columns:")
print(unrelated_patch_df.columns.tolist())

print("\nNegative patch columns:")
print(negative_patch_df.columns.tolist())

print("\nUnrelated donor-strength columns:")
print(unrelated_strength_df.columns.tolist())

assert len(matched_patch_df) == 189
assert len(unrelated_patch_df) == 189
assert len(negative_patch_df) == 189
assert len(negative_pool_df) == 189
assert len(unrelated_strength_df) == 189

assert matched_patch_df["pair_index"].is_unique
assert unrelated_patch_df[
    "recipient_pair_index"
].is_unique
assert negative_patch_df[
    "recipient_pair_index"
].is_unique

print("\nNotebook 4 data validation passed.")

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.9.0. This might lead to breaking code or in

Loaded rows:
 matched promoter patches: 189
 unrelated promoter patches: 189
 non-promoter patches: 189
 negative donor pool: 189
 unrelated donor strength: 189
 baseline clean/corrupt scores: 291

Matched patch columns:
['pair_index', 'original_cohort_index', 'clean_logit', 'corrupt_logit', 'patched_logit', 'clean_corrupt_gap', 'rescue_amount', 'recovery_fraction', 'clean_probability', 'corrupt_probability', 'patched_probability', 'number_patched_tokens', 'mean_branch_change_norm']

Unrelated patch columns:
['recipient_pair_index', 'unrelated_donor_index', 'clean_logit', 'corrupt_logit', 'unrelated_patched_logit', 'clean_corrupt_gap', 'unrelated_rescue_amount', 'unrelated_recovery_fraction', 'unrelated_patched_probability', 'mean_branch_change_norm']

Negative patch columns:
['recipient_pair_index', 'negative_donor_index', 'negative_donor_probe_logit', 'negative_donor_probe_probability', 'clean_logit', 'corrupt_logit', 'patched_logit', 'clean_corrupt_gap', 'rescue_amount', 'recovery_f

# Cell 4 — Reconstruct exact donor sequences and verify index mappings

In [4]:
SEQUENCE_PAIR_PATH = (
    CAUSAL_DIR
    / "matched_clean_dinucleotide_corrupt_sequences.csv"
)

sequence_pair_df = pd.read_csv(
    SEQUENCE_PAIR_PATH
)

print("Sequence-pair rows:", len(sequence_pair_df))
print("\nSequence-pair columns:")
print(sequence_pair_df.columns.tolist())

print("\nBaseline-score columns:")
print(baseline_scores_df.columns.tolist())

display(sequence_pair_df.head())
display(baseline_scores_df.head())


# Identify likely sequence columns robustly.
clean_sequence_candidates = [
    "clean_sequence",
    "original_sequence",
    "sequence",
    "recognized_sequence",
]

corrupt_sequence_candidates = [
    "corrupt_sequence",
    "shuffled_sequence",
    "dinucleotide_corrupt_sequence",
]

clean_sequence_column = next(
    (
        column
        for column in clean_sequence_candidates
        if column in sequence_pair_df.columns
    ),
    None,
)

corrupt_sequence_column = next(
    (
        column
        for column in corrupt_sequence_candidates
        if column in sequence_pair_df.columns
    ),
    None,
)

if clean_sequence_column is None:
    raise KeyError(
        "Could not identify the clean-sequence column. "
        f"Available columns: {sequence_pair_df.columns.tolist()}"
    )

if corrupt_sequence_column is None:
    raise KeyError(
        "Could not identify the corrupt-sequence column. "
        f"Available columns: {sequence_pair_df.columns.tolist()}"
    )

print("\nDetected clean sequence column:", clean_sequence_column)
print("Detected corrupt sequence column:", corrupt_sequence_column)


# Determine which column identifies the original 291-sequence cohort.
cohort_index_candidates = [
    "original_cohort_index",
    "cohort_index",
    "recognized_index",
    "sequence_index",
]

sequence_pair_index_column = next(
    (
        column
        for column in cohort_index_candidates
        if column in sequence_pair_df.columns
    ),
    None,
)

if sequence_pair_index_column is None:
    # If the saved file is already ordered like the 291-sequence cohort,
    # create the index explicitly.
    sequence_pair_df = sequence_pair_df.copy()
    sequence_pair_df["original_cohort_index"] = np.arange(
        len(sequence_pair_df)
    )
    sequence_pair_index_column = "original_cohort_index"

print(
    "Detected sequence-pair cohort index column:",
    sequence_pair_index_column,
)


# Build the 189 matched promoter donor table using the exact
# original cohort indices saved in the matched patch results.
matched_promoter_donor_df = (
    matched_patch_df[
        [
            "pair_index",
            "original_cohort_index",
            "clean_logit",
            "clean_probability",
            "rescue_amount",
            "recovery_fraction",
        ]
    ]
    .merge(
        sequence_pair_df[
            [
                sequence_pair_index_column,
                clean_sequence_column,
                corrupt_sequence_column,
            ]
        ],
        left_on="original_cohort_index",
        right_on=sequence_pair_index_column,
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            clean_sequence_column: "donor_sequence",
            corrupt_sequence_column: "recipient_corrupt_sequence",
            "clean_logit": "donor_probe_logit",
            "clean_probability": "donor_probe_probability",
            "rescue_amount": "matched_rescue_amount",
            "recovery_fraction": "matched_recovery_fraction",
        }
    )
)

assert (
    matched_promoter_donor_df["donor_sequence"]
    .notna()
    .all()
)

assert len(matched_promoter_donor_df) == 189


# Build the unrelated promoter donor table.
# donor_pair_index indexes the same 189-row rescue cohort.
promoter_sequence_lookup = (
    matched_promoter_donor_df
    .set_index("pair_index")
)

unrelated_promoter_donor_df = (
    unrelated_strength_df.copy()
)

unrelated_promoter_donor_df[
    "donor_sequence"
] = unrelated_promoter_donor_df[
    "donor_pair_index"
].map(
    promoter_sequence_lookup[
        "donor_sequence"
    ]
)

assert (
    unrelated_promoter_donor_df[
        "donor_sequence"
    ]
    .notna()
    .all()
)


# Build the non-promoter donor table.
non_promoter_donor_df = (
    negative_pool_df.copy()
    .rename(
        columns={
            "sequence": "donor_sequence",
            "probe_logit": "donor_probe_logit",
            "probe_probability": "donor_probe_probability",
        }
    )
)

non_promoter_donor_df[
    "rescue_amount"
] = (
    negative_patch_df
    .sort_values("negative_donor_index")[
        "rescue_amount"
    ]
    .to_numpy()
)

non_promoter_donor_df[
    "recovery_fraction"
] = (
    negative_patch_df
    .sort_values("negative_donor_index")[
        "recovery_fraction"
    ]
    .to_numpy()
)


print("\nMatched promoter donor table:")
display(matched_promoter_donor_df.head())

print("\nUnrelated promoter donor table:")
display(
    unrelated_promoter_donor_df[
        [
            "recipient_pair_index",
            "donor_pair_index",
            "donor_sequence",
            "donor_clean_logit",
            "unrelated_rescue_amount",
        ]
    ].head()
)

print("\nNon-promoter donor table:")
display(non_promoter_donor_df.head())


print("\nSequence-length checks:")

for name, dataframe in [
    ("matched promoter", matched_promoter_donor_df),
    ("unrelated promoter", unrelated_promoter_donor_df),
    ("non-promoter", non_promoter_donor_df),
]:
    lengths = dataframe["donor_sequence"].str.len()

    print(
        name,
        "rows =", len(dataframe),
        "min length =", lengths.min(),
        "max length =", lengths.max(),
        "unique lengths =", sorted(lengths.unique().tolist()),
    )


assert len(unrelated_promoter_donor_df) == 189
assert len(non_promoter_donor_df) == 189

print("\nDonor-sequence reconstruction passed.")

Sequence-pair rows: 291

Sequence-pair columns:
['cohort_index', 'original_sequence_index', 'clean_sequence', 'corrupt_sequence', 'clean_sequence_length']

Baseline-score columns:
['cohort_index', 'original_sequence_index', 'clean_sequence', 'corrupt_sequence', 'clean_sequence_length', 'clean_logit', 'corrupt_logit', 'clean_corrupt_gap', 'clean_probability', 'corrupt_probability', 'valid_rescue_pair']


,cohort_index,original_sequence_index,clean_sequence,corrupt_sequence,clean_sequence_length
0,0,1,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CGCGGACCCGGCGCCAGCCCGGCCGCCCCTGCAGACCGCCTGCATC...,70
1,1,2,ACCCTGGGGCTGCGCGCTGGGGCGCGAACAGCCAGAGCGTCGGCGC...,AGCACAGAGCAACCCGTTCGGCGGGCGGCTGACGCCGCTCACCCTG...,70
2,2,3,GGTCTTTAACACCGCCCAGCGCACATGTCGGGGGAGGCCTGGCAGC...,GCAGGGGGCAGTAGCGGGGGCGGCTGTCACAGCTGCAGCACACTCC...,70
3,3,5,AGGCGGAGGAGCTTGGCGAGGGGAGGCGCAGGCTCGGAAAGGCGCG...,AGGAGCTCCCAGGAGGGCGGGCGAAGCGAGCAAGGAGCTCGATGAG...,70
4,4,6,ATCCTATAAAGGCAGTAGCTCCAAGCACATTACAGAGGGACCCAAC...,ATAGGGAAACATTCCTCTAAGCTTACAAGCAGGCCACCACTCCAGT...,70


,cohort_index,original_sequence_index,clean_sequence,corrupt_sequence,clean_sequence_length,clean_logit,corrupt_logit,clean_corrupt_gap,clean_probability,corrupt_probability,valid_rescue_pair
0,0,1,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CGCGGACCCGGCGCCAGCCCGGCCGCCCCTGCAGACCGCCTGCATC...,70,5.393593,3.291286,2.102307,0.995475,0.964129,True
1,1,2,ACCCTGGGGCTGCGCGCTGGGGCGCGAACAGCCAGAGCGTCGGCGC...,AGCACAGAGCAACCCGTTCGGCGGGCGGCTGACGCCGCTCACCCTG...,70,5.007429,3.685326,1.322102,0.993356,0.975525,True
2,2,3,GGTCTTTAACACCGCCCAGCGCACATGTCGGGGGAGGCCTGGCAGC...,GCAGGGGGCAGTAGCGGGGGCGGCTGTCACAGCTGCAGCACACTCC...,70,6.289941,-0.207787,6.497728,0.998149,0.448239,True
3,3,5,AGGCGGAGGAGCTTGGCGAGGGGAGGCGCAGGCTCGGAAAGGCGCG...,AGGAGCTCCCAGGAGGGCGGGCGAAGCGAGCAAGGAGCTCGATGAG...,70,4.211724,1.402079,2.809645,0.985396,0.802514,True
4,4,6,ATCCTATAAAGGCAGTAGCTCCAAGCACATTACAGAGGGACCCAAC...,ATAGGGAAACATTCCTCTAAGCTTACAAGCAGGCCACCACTCCAGT...,70,1.968987,4.681609,-2.712622,0.877502,0.990821,False



Detected clean sequence column: clean_sequence
Detected corrupt sequence column: corrupt_sequence
Detected sequence-pair cohort index column: cohort_index

Matched promoter donor table:


,pair_index,original_cohort_index,donor_probe_logit,donor_probe_probability,matched_rescue_amount,matched_recovery_fraction,cohort_index,donor_sequence,recipient_corrupt_sequence
0,0,0,5.393593,0.995475,12.590756,5.989018,0,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CGCGGACCCGGCGCCAGCCCGGCCGCCCCTGCAGACCGCCTGCATC...
1,1,1,5.007429,0.993356,13.095290,9.904899,1,ACCCTGGGGCTGCGCGCTGGGGCGCGAACAGCCAGAGCGTCGGCGC...,AGCACAGAGCAACCCGTTCGGCGGGCGGCTGACGCCGCTCACCCTG...
2,2,2,6.289941,0.998149,25.938336,3.991909,2,GGTCTTTAACACCGCCCAGCGCACATGTCGGGGGAGGCCTGGCAGC...,GCAGGGGGCAGTAGCGGGGGCGGCTGTCACAGCTGCAGCACACTCC...
3,3,3,4.211724,0.985396,11.174763,3.977286,3,AGGCGGAGGAGCTTGGCGAGGGGAGGCGCAGGCTCGGAAAGGCGCG...,AGGAGCTCCCAGGAGGGCGGGCGAAGCGAGCAAGGAGCTCGATGAG...
4,4,6,2.431718,0.919214,13.419673,1.395292,6,AAACCGAGTGACTCTGATATTTTTCCTTGGGGGCGTAACCTTCGCT...,AAGATGGCTTTTGCCTCCGAATGACTTGGCCCTGTGGGAAATTCCT...



Unrelated promoter donor table:


,recipient_pair_index,donor_pair_index,donor_sequence,donor_clean_logit,unrelated_rescue_amount
0,0,2,GGTCTTTAACACCGCCCAGCGCACATGTCGGGGGAGGCCTGGCAGC...,6.289941,20.398334
1,1,45,TTGGGTGTGGCAGGCACTGTGTTGACATATCTCATTTCTGAAAGAG...,4.505828,-3.743812
2,2,183,GGGGTCTCACAGCGACTGCGCGGACGGGTTCCTGAGTGGAACATGG...,1.111230,1.627896
3,3,105,GCTTGTGGGGTCCTTGGGTGGCAGGGGCTGAGCATGGGAAAGTGAG...,3.132630,-3.178125
4,4,115,TCTACGCTGGGGGCGTGGCCTGACTGCGCGGCCAGACGCCAGCGCC...,1.296792,23.939525



Non-promoter donor table:


,negative_donor_index,donor_sequence,donor_probe_logit,donor_probe_probability,rescue_amount,recovery_fraction
0,0,CCAGGAATTATTTCGCCTTATTCCAATTCATAGAGACCAGAGGCGA...,2.096603,0.890573,11.763066,5.595312
1,1,TGAGTCATACGGGCCCCTGCCCCGTCCCCGCCGCTCCGTTGAGCGA...,-6.911253,0.000996,-6.318379,-4.779039
2,2,TTCTCCTTTCGAGGGATCCACCTACTCCTTCTCAGGTTCCAGCTAA...,1.905359,0.870497,-3.662704,-0.563690
3,3,CGCCCGCCCGCGCTTCCGTCCTGTTAATGCCGTGTCACAACTTCGG...,-8.689655,0.000168,4.971352,1.769388
4,4,GACCAGGCCTCGGAGCGCGGCGGGAACAGATCCATACGGGGAAGAG...,-3.454822,0.030625,8.114892,0.843735



Sequence-length checks:
matched promoter rows = 189 min length = 70 max length = 70 unique lengths = [70]
unrelated promoter rows = 189 min length = 70 max length = 70 unique lengths = [70]
non-promoter rows = 189 min length = 70 max length = 70 unique lengths = [70]

Donor-sequence reconstruction passed.


#Cell 5 — calculate sequence composition and motif features

In [5]:
from collections import Counter
from math import log2
import re


def shannon_entropy(sequence):
    counts = Counter(sequence)
    length = len(sequence)

    return -sum(
        (count / length) * log2(count / length)
        for count in counts.values()
        if count > 0
    )


def longest_homopolymer_run(sequence):
    longest = 1
    current = 1

    for index in range(1, len(sequence)):
        if sequence[index] == sequence[index - 1]:
            current += 1
            longest = max(longest, current)
        else:
            current = 1

    return longest


def overlapping_count(sequence, motif):
    return sum(
        1
        for index in range(
            len(sequence) - len(motif) + 1
        )
        if sequence[index:index + len(motif)] == motif
    )


def minimum_hamming_distance_to_motif(
    sequence,
    motif,
):
    motif_length = len(motif)

    distances = []

    for index in range(
        len(sequence) - motif_length + 1
    ):
        window = sequence[
            index:index + motif_length
        ]

        distance = sum(
            base_a != base_b
            for base_a, base_b in zip(
                window,
                motif,
            )
        )

        distances.append(distance)

    return min(distances)


def compute_sequence_features(sequence):
    sequence = sequence.upper()
    length = len(sequence)

    nucleotide_counts = Counter(sequence)

    gc_count = (
        nucleotide_counts.get("G", 0)
        + nucleotide_counts.get("C", 0)
    )

    at_count = (
        nucleotide_counts.get("A", 0)
        + nucleotide_counts.get("T", 0)
    )

    dinucleotides = [
        sequence[index:index + 2]
        for index in range(length - 1)
    ]

    dinucleotide_counts = Counter(
        dinucleotides
    )

    cpg_count = dinucleotide_counts.get(
        "CG",
        0,
    )

    gpc_count = dinucleotide_counts.get(
        "GC",
        0,
    )

    # Common promoter-associated sequence patterns.
    tata_exact_count = (
        overlapping_count(sequence, "TATAAA")
        + overlapping_count(sequence, "TATAAT")
    )

    caat_exact_count = overlapping_count(
        sequence,
        "CCAAT",
    )

    gc_box_exact_count = (
        overlapping_count(sequence, "GGGCGG")
        + overlapping_count(sequence, "GGGCGG")
        + overlapping_count(sequence, "CCGCCC")
    )

    initiator_patterns = [
        "YYANWYY",
    ]

    # Approximate human initiator consensus:
    # YYANWYY, represented directly by a regex.
    initiator_regex = re.compile(
        r"[CT][CT]A[ACGT][AT][CT][CT]"
    )

    initiator_matches = sum(
        1
        for index in range(length - 7 + 1)
        if initiator_regex.fullmatch(
            sequence[index:index + 7]
        )
    )

    return {
        "sequence_length": length,

        "a_fraction": (
            nucleotide_counts.get("A", 0)
            / length
        ),
        "c_fraction": (
            nucleotide_counts.get("C", 0)
            / length
        ),
        "g_fraction": (
            nucleotide_counts.get("G", 0)
            / length
        ),
        "t_fraction": (
            nucleotide_counts.get("T", 0)
            / length
        ),

        "gc_fraction": gc_count / length,
        "at_fraction": at_count / length,

        "cpg_count": cpg_count,
        "cpg_density": (
            cpg_count / (length - 1)
        ),
        "gpc_count": gpc_count,
        "gpc_density": (
            gpc_count / (length - 1)
        ),

        "cpg_observed_expected": (
            (
                cpg_count * length
                / max(
                    1,
                    nucleotide_counts.get("C", 0)
                    * nucleotide_counts.get("G", 0),
                )
            )
        ),

        "sequence_entropy": (
            shannon_entropy(sequence)
        ),

        "longest_homopolymer_run": (
            longest_homopolymer_run(sequence)
        ),

        "tata_exact_count": tata_exact_count,
        "tata_present": int(
            tata_exact_count > 0
        ),
        "tata_min_hamming": min(
            minimum_hamming_distance_to_motif(
                sequence,
                "TATAAA",
            ),
            minimum_hamming_distance_to_motif(
                sequence,
                "TATAAT",
            ),
        ),

        "caat_exact_count": caat_exact_count,
        "caat_present": int(
            caat_exact_count > 0
        ),
        "caat_min_hamming": (
            minimum_hamming_distance_to_motif(
                sequence,
                "CCAAT",
            )
        ),

        "gc_box_exact_count": (
            gc_box_exact_count
        ),
        "gc_box_present": int(
            gc_box_exact_count > 0
        ),
        "gc_box_min_hamming": min(
            minimum_hamming_distance_to_motif(
                sequence,
                "GGGCGG",
            ),
            minimum_hamming_distance_to_motif(
                sequence,
                "CCGCCC",
            ),
        ),

        "initiator_match_count": (
            initiator_matches
        ),
        "initiator_present": int(
            initiator_matches > 0
        ),
    }


def add_sequence_features(
    dataframe,
    sequence_column="donor_sequence",
):
    feature_rows = [
        compute_sequence_features(sequence)
        for sequence in dataframe[
            sequence_column
        ]
    ]

    feature_df = pd.DataFrame(
        feature_rows,
        index=dataframe.index,
    )

    return pd.concat(
        [
            dataframe.reset_index(drop=True),
            feature_df.reset_index(drop=True),
        ],
        axis=1,
    )


matched_promoter_feature_df = (
    add_sequence_features(
        matched_promoter_donor_df
    )
)

unrelated_promoter_feature_df = (
    add_sequence_features(
        unrelated_promoter_donor_df
    )
)

non_promoter_feature_df = (
    add_sequence_features(
        non_promoter_donor_df
    )
)


# Add standardized column names for later combined analyses.
matched_promoter_feature_df[
    "donor_class"
] = "matched promoter"

matched_promoter_feature_df[
    "analysis_rescue_amount"
] = (
    matched_promoter_feature_df[
        "matched_rescue_amount"
    ]
)


unrelated_promoter_feature_df[
    "donor_class"
] = "unrelated promoter"

unrelated_promoter_feature_df[
    "analysis_rescue_amount"
] = (
    unrelated_promoter_feature_df[
        "unrelated_rescue_amount"
    ]
)

unrelated_promoter_feature_df[
    "donor_probe_logit"
] = (
    unrelated_promoter_feature_df[
        "donor_clean_logit"
    ]
)

unrelated_promoter_feature_df[
    "donor_probe_probability"
] = (
    unrelated_promoter_feature_df[
        "donor_clean_probability"
    ]
)


non_promoter_feature_df[
    "donor_class"
] = "non-promoter"

non_promoter_feature_df[
    "analysis_rescue_amount"
] = (
    non_promoter_feature_df[
        "rescue_amount"
    ]
)


common_feature_columns = [
    "donor_class",
    "donor_sequence",
    "donor_probe_logit",
    "donor_probe_probability",
    "analysis_rescue_amount",
    "sequence_length",
    "a_fraction",
    "c_fraction",
    "g_fraction",
    "t_fraction",
    "gc_fraction",
    "at_fraction",
    "cpg_count",
    "cpg_density",
    "gpc_count",
    "gpc_density",
    "cpg_observed_expected",
    "sequence_entropy",
    "longest_homopolymer_run",
    "tata_exact_count",
    "tata_present",
    "tata_min_hamming",
    "caat_exact_count",
    "caat_present",
    "caat_min_hamming",
    "gc_box_exact_count",
    "gc_box_present",
    "gc_box_min_hamming",
    "initiator_match_count",
    "initiator_present",
]


combined_donor_feature_df = pd.concat(
    [
        matched_promoter_feature_df[
            common_feature_columns
        ],
        unrelated_promoter_feature_df[
            common_feature_columns
        ],
        non_promoter_feature_df[
            common_feature_columns
        ],
    ],
    ignore_index=True,
)


print("Combined donor-feature rows:")
print(len(combined_donor_feature_df))

print("\nRows by donor class:")
display(
    combined_donor_feature_df[
        "donor_class"
    ]
    .value_counts()
    .rename_axis("donor_class")
    .reset_index(name="count")
)

print("\nFeature preview:")
display(
    combined_donor_feature_df.head(10)
)

print("\nBasic feature checks:")
print(
    "Missing values:",
    combined_donor_feature_df.isna().sum().sum(),
)

print(
    "GC range:",
    combined_donor_feature_df[
        "gc_fraction"
    ].min(),
    "to",
    combined_donor_feature_df[
        "gc_fraction"
    ].max(),
)

print(
    "CpG density range:",
    combined_donor_feature_df[
        "cpg_density"
    ].min(),
    "to",
    combined_donor_feature_df[
        "cpg_density"
    ].max(),
)

print(
    "TATA-present sequences:",
    combined_donor_feature_df[
        "tata_present"
    ].sum(),
)

print(
    "GC-box-present sequences:",
    combined_donor_feature_df[
        "gc_box_present"
    ].sum(),
)

print(
    "Initiator-present sequences:",
    combined_donor_feature_df[
        "initiator_present"
    ].sum(),
)


matched_promoter_feature_df.to_csv(
    CAUSAL_DIR
    / "notebook5_matched_promoter_features.csv",
    index=False,
)

unrelated_promoter_feature_df.to_csv(
    CAUSAL_DIR
    / "notebook5_unrelated_promoter_features.csv",
    index=False,
)

non_promoter_feature_df.to_csv(
    CAUSAL_DIR
    / "notebook5_non_promoter_features.csv",
    index=False,
)

combined_donor_feature_df.to_csv(
    CAUSAL_DIR
    / "notebook5_combined_donor_features.csv",
    index=False,
)

print(
    "\nSaved Notebook 5 feature tables to:",
    CAUSAL_DIR,
)

Combined donor-feature rows:
567

Rows by donor class:


,donor_class,count
0,matched promoter,189
1,unrelated promoter,189
2,non-promoter,189



Feature preview:


,donor_class,donor_sequence,donor_probe_logit,donor_probe_probability,analysis_rescue_amount,sequence_length,a_fraction,c_fraction,g_fraction,t_fraction,...,tata_present,tata_min_hamming,caat_exact_count,caat_present,caat_min_hamming,gc_box_exact_count,gc_box_present,gc_box_min_hamming,initiator_match_count,initiator_present
0,matched promoter,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,5.393593,0.995475,12.590756,70,0.114286,0.514286,0.314286,0.057143,...,0,4,0,0,2,0,0,1,0,0
1,matched promoter,ACCCTGGGGCTGCGCGCTGGGGCGCGAACAGCCAGAGCGTCGGCGC...,5.007429,0.993356,13.095290,70,0.171429,0.371429,0.357143,0.100000,...,0,3,0,0,2,0,0,1,0,0
2,matched promoter,GGTCTTTAACACCGCCCAGCGCACATGTCGGGGGAGGCCTGGCAGC...,6.289941,0.998149,25.938336,70,0.185714,0.314286,0.371429,0.128571,...,0,2,0,0,2,1,1,0,0,0
3,matched promoter,AGGCGGAGGAGCTTGGCGAGGGGAGGCGCAGGCTCGGAAAGGCGCG...,4.211724,0.985396,11.174763,70,0.171429,0.271429,0.442857,0.114286,...,0,3,0,0,1,0,0,1,0,0
4,matched promoter,AAACCGAGTGACTCTGATATTTTTCCTTGGGGGCGTAACCTTCGCT...,2.431718,0.919214,13.419673,70,0.185714,0.257143,0.228571,0.328571,...,0,2,0,0,2,0,0,1,0,0
5,matched promoter,AGCCAATCAGCGGCCGCCTCTTGCGCGCGCCCCACCGACCAAAATG...,6.169904,0.997913,6.243320,70,0.142857,0.371429,0.342857,0.142857,...,0,2,1,1,0,0,0,1,0,0
6,matched promoter,TTCTTCCGCCCCGCTGACGCCCCAGGTGGAGGGAGCATGGCCACCT...,10.836474,0.999980,15.166216,70,0.171429,0.471429,0.228571,0.128571,...,0,3,0,0,1,1,1,0,0,0
7,matched promoter,GGTCGGCTTGGCTGAGCGGGGGCGGTGCTGGGCAGGGCGGCGGCCG...,21.488038,1.000000,10.474314,70,0.042857,0.371429,0.442857,0.142857,...,0,4,0,0,2,4,1,0,0,0
8,matched promoter,CCTGCCCACCCCGCCCCCGCGGCCGAGCCCGGGAGTCGAGTGGGAG...,17.036306,1.000000,30.592123,70,0.085714,0.442857,0.414286,0.057143,...,0,3,0,0,2,1,1,0,0,0
9,matched promoter,CAGCAAGAGGGGGAGTCAGAAGAAAAACAGATCAGACCAGAGCAGA...,8.861694,0.999858,17.136952,70,0.385714,0.242857,0.271429,0.100000,...,0,2,0,0,1,0,0,2,1,1



Basic feature checks:
Missing values: 0
GC range: 0.2571428571428571 to 0.8857142857142857
CpG density range: 0.0 to 0.2463768115942029
TATA-present sequences: 11
GC-box-present sequences: 106
Initiator-present sequences: 155

Saved Notebook 5 feature tables to: /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_causal_interventions


# Cell 6 — Correct GC-box counts and compare independent donor classes

In [6]:
def corrected_gc_box_features(sequence):
    sequence = sequence.upper()

    gc_box_exact_count = (
        overlapping_count(sequence, "GGGCGG")
        + overlapping_count(sequence, "CCGCCC")
    )

    return {
        "gc_box_exact_count": gc_box_exact_count,
        "gc_box_present": int(
            gc_box_exact_count > 0
        ),
        "gc_box_min_hamming": min(
            minimum_hamming_distance_to_motif(
                sequence,
                "GGGCGG",
            ),
            minimum_hamming_distance_to_motif(
                sequence,
                "CCGCCC",
            ),
        ),
    }


def replace_gc_box_features(dataframe):
    corrected_rows = pd.DataFrame(
        [
            corrected_gc_box_features(sequence)
            for sequence in dataframe[
                "donor_sequence"
            ]
        ],
        index=dataframe.index,
    )

    dataframe = dataframe.copy()

    for column in corrected_rows.columns:
        dataframe[column] = (
            corrected_rows[column].to_numpy()
        )

    return dataframe


matched_promoter_feature_df = (
    replace_gc_box_features(
        matched_promoter_feature_df
    )
)

unrelated_promoter_feature_df = (
    replace_gc_box_features(
        unrelated_promoter_feature_df
    )
)

non_promoter_feature_df = (
    replace_gc_box_features(
        non_promoter_feature_df
    )
)


# Use only one copy of the promoter sequence set for
# biological class comparisons.
promoter_class_df = (
    matched_promoter_feature_df.copy()
)

promoter_class_df[
    "biological_class"
] = "promoter"

negative_class_df = (
    non_promoter_feature_df.copy()
)

negative_class_df[
    "biological_class"
] = "non-promoter"


independent_class_feature_df = pd.concat(
    [
        promoter_class_df,
        negative_class_df,
    ],
    ignore_index=True,
)


continuous_features = [
    "gc_fraction",
    "cpg_density",
    "cpg_observed_expected",
    "gpc_density",
    "sequence_entropy",
    "longest_homopolymer_run",
    "tata_min_hamming",
    "caat_min_hamming",
    "gc_box_min_hamming",
    "tata_exact_count",
    "caat_exact_count",
    "gc_box_exact_count",
    "initiator_match_count",
]


binary_features = [
    "tata_present",
    "caat_present",
    "gc_box_present",
    "initiator_present",
]


# Summarize continuous features by biological class.
continuous_class_summary = (
    independent_class_feature_df
    .groupby("biological_class")[
        continuous_features
    ]
    .agg(["mean", "median", "std"])
)

print(
    "Continuous feature summary by class:"
)

display(continuous_class_summary)


# Summarize motif prevalence.
binary_class_summary_rows = []

for feature in binary_features:
    for biological_class, class_df in (
        independent_class_feature_df
        .groupby("biological_class")
    ):
        binary_class_summary_rows.append({
            "feature": feature,
            "biological_class": (
                biological_class
            ),
            "number_present": int(
                class_df[feature].sum()
            ),
            "number_total": len(class_df),
            "fraction_present": (
                class_df[feature].mean()
            ),
        })

binary_class_summary = pd.DataFrame(
    binary_class_summary_rows
)

print("\nBinary motif prevalence:")
display(binary_class_summary)


# Mann–Whitney tests for continuous features.
from scipy.stats import mannwhitneyu
from scipy.stats import fisher_exact

continuous_test_rows = []

promoter_rows = (
    independent_class_feature_df[
        independent_class_feature_df[
            "biological_class"
        ] == "promoter"
    ]
)

negative_rows = (
    independent_class_feature_df[
        independent_class_feature_df[
            "biological_class"
        ] == "non-promoter"
    ]
)

for feature in continuous_features:
    promoter_values = (
        promoter_rows[feature].to_numpy()
    )

    negative_values = (
        negative_rows[feature].to_numpy()
    )

    statistic, p_value = mannwhitneyu(
        promoter_values,
        negative_values,
        alternative="two-sided",
    )

    continuous_test_rows.append({
        "feature": feature,
        "promoter_mean": (
            promoter_values.mean()
        ),
        "non_promoter_mean": (
            negative_values.mean()
        ),
        "mean_difference": (
            promoter_values.mean()
            - negative_values.mean()
        ),
        "mann_whitney_u": statistic,
        "p_value": p_value,
    })

continuous_feature_tests = pd.DataFrame(
    continuous_test_rows
).sort_values("p_value")


# Fisher exact tests for binary motifs.
binary_test_rows = []

for feature in binary_features:
    promoter_present = int(
        promoter_rows[feature].sum()
    )

    promoter_absent = (
        len(promoter_rows)
        - promoter_present
    )

    negative_present = int(
        negative_rows[feature].sum()
    )

    negative_absent = (
        len(negative_rows)
        - negative_present
    )

    odds_ratio, p_value = fisher_exact(
        [
            [
                promoter_present,
                promoter_absent,
            ],
            [
                negative_present,
                negative_absent,
            ],
        ],
        alternative="two-sided",
    )

    binary_test_rows.append({
        "feature": feature,
        "promoter_fraction": (
            promoter_present
            / len(promoter_rows)
        ),
        "non_promoter_fraction": (
            negative_present
            / len(negative_rows)
        ),
        "odds_ratio": odds_ratio,
        "p_value": p_value,
    })

binary_feature_tests = pd.DataFrame(
    binary_test_rows
).sort_values("p_value")


print(
    "\nContinuous promoter-versus-non-promoter tests:"
)

display(continuous_feature_tests)


print(
    "\nBinary motif promoter-versus-non-promoter tests:"
)

display(binary_feature_tests)


print(
    "\nCorrected GC-box-positive counts:"
)

print(
    "Promoters:",
    int(promoter_rows["gc_box_present"].sum()),
    "/",
    len(promoter_rows),
)

print(
    "Non-promoters:",
    int(negative_rows["gc_box_present"].sum()),
    "/",
    len(negative_rows),
)


matched_promoter_feature_df.to_csv(
    CAUSAL_DIR
    / "notebook5_matched_promoter_features.csv",
    index=False,
)

unrelated_promoter_feature_df.to_csv(
    CAUSAL_DIR
    / "notebook5_unrelated_promoter_features.csv",
    index=False,
)

non_promoter_feature_df.to_csv(
    CAUSAL_DIR
    / "notebook5_non_promoter_features.csv",
    index=False,
)

continuous_feature_tests.to_csv(
    CAUSAL_DIR
    / "notebook5_promoter_vs_nonpromoter_continuous_tests.csv",
    index=False,
)

binary_feature_tests.to_csv(
    CAUSAL_DIR
    / "notebook5_promoter_vs_nonpromoter_binary_tests.csv",
    index=False,
)

print(
    "\nSaved corrected feature analyses to:",
    CAUSAL_DIR,
)

Continuous feature summary by class:


gc_fraction                     cpg_density            \
                        mean    median       std        mean    median   
biological_class                                                         
non-promoter        0.543159  0.542857  0.086608    0.063339  0.057971   
promoter            0.643764  0.657143  0.128602    0.080209  0.072464   

                           cpg_observed_expected                      \
                       std                  mean    median       std   
biological_class                                                       
non-promoter      0.035217              0.828377  0.795455  0.397836   
promoter          0.060212              0.673228  0.760870  0.395597   

                 gpc_density  ... tata_exact_count caat_exact_count         \
                        mean  ...              std             mean median   
biological_class              ...                                            
non-promoter        0.076681  ...         0.247013         0.068783    0.0   
promoter            0.112721  ...         0.072739         0.026455    0.0   

                           gc_box_exact_count                   \
                       std               mean median       std   
biological_class                                                 
non-promoter      0.273918           0.095238    0.0  0.328486   
promoter          0.160910           0.312169    0.0  0.629851   

                 initiator_match_count                   
                                  mean median       std  
biological_class                                         
non-promoter                  0.439153    0.0  0.654568  
promoter                      0.269841    0.0  0.522057  

[2 rows x 39 columns]


Binary motif prevalence:


,feature,biological_class,number_present,number_total,fraction_present
0,tata_present,non-promoter,9,189,0.047619
1,tata_present,promoter,1,189,0.005291
2,caat_present,non-promoter,12,189,0.063492
3,caat_present,promoter,5,189,0.026455
4,gc_box_present,non-promoter,16,189,0.084656
5,gc_box_present,promoter,45,189,0.238095
6,initiator_present,non-promoter,67,189,0.354497
7,initiator_present,promoter,44,189,0.232804



Continuous promoter-versus-non-promoter tests:


,feature,promoter_mean,non_promoter_mean,mean_difference,mann_whitney_u,p_value
4,sequence_entropy,1.834495,1.941529,-0.107034,6975.0,1.206078e-24
6,tata_min_hamming,2.492063,1.746032,0.746032,26197.5,1.371204e-17
0,gc_fraction,0.643764,0.543159,0.100605,26793.0,3.926690e-17
3,gpc_density,0.112721,0.076681,0.036040,24765.5,6.520475e-11
7,caat_min_hamming,1.661376,1.328042,0.333333,22834.0,1.541708e-07
11,gc_box_exact_count,0.312169,0.095238,0.216931,20647.0,4.031118e-05
8,gc_box_min_hamming,1.042328,1.338624,-0.296296,14214.5,1.686440e-04
2,cpg_observed_expected,0.673228,0.828377,-0.155149,14605.0,2.172566e-03
12,initiator_match_count,0.269841,0.439153,-0.169312,15588.0,7.238956e-03
9,tata_exact_count,0.005291,0.052910,-0.047619,17104.0,1.045253e-02



Binary motif promoter-versus-non-promoter tests:


,feature,promoter_fraction,non_promoter_fraction,odds_ratio,p_value
2,gc_box_present,0.238095,0.084656,3.378906,0.000071
3,initiator_present,0.232804,0.354497,0.552548,0.012782
0,tata_present,0.005291,0.047619,0.106383,0.019879
1,caat_present,0.026455,0.063492,0.400815,0.134389



Corrected GC-box-positive counts:
Promoters: 45 / 189
Non-promoters: 16 / 189

Saved corrected feature analyses to: /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_causal_interventions


# Cell 7 — Test which donor features predict MLP rescue
# while accounting for donor probe strength

In [7]:
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from statsmodels.stats.multitest import multipletests


# Use the unrelated-promoter intervention because donor and recipient
# are different sequences. This is the cleanest dataset for asking
# which donor properties predict transferred rescue.
rescue_feature_df = (
    unrelated_promoter_feature_df.copy()
)


candidate_features = [
    "gc_fraction",
    "cpg_density",
    "cpg_observed_expected",
    "gpc_density",
    "sequence_entropy",
    "longest_homopolymer_run",
    "tata_min_hamming",
    "caat_min_hamming",
    "gc_box_min_hamming",
    "tata_exact_count",
    "caat_exact_count",
    "gc_box_exact_count",
    "initiator_match_count",
    "tata_present",
    "caat_present",
    "gc_box_present",
    "initiator_present",
]


# Recipient variables may affect how much rescue is possible.
recipient_covariates = [
    "donor_probe_logit",
    "recipient_corrupt_logit",
    "recipient_gap",
]


def residualize(values, covariate_matrix):
    model = LinearRegression()
    model.fit(covariate_matrix, values)

    return values - model.predict(
        covariate_matrix
    )


# Raw correlations and partial correlations.
feature_rescue_rows = []

rescue_values = (
    rescue_feature_df[
        "unrelated_rescue_amount"
    ].to_numpy(dtype=float)
)

covariate_matrix = (
    rescue_feature_df[
        recipient_covariates
    ].to_numpy(dtype=float)
)

residual_rescue = residualize(
    rescue_values,
    covariate_matrix,
)


for feature in candidate_features:
    feature_values = (
        rescue_feature_df[feature]
        .to_numpy(dtype=float)
    )

    raw_result = spearmanr(
        feature_values,
        rescue_values,
    )

    # Remove linear dependence on donor probe logit and
    # recipient-level opportunity for rescue.
    residual_feature = residualize(
        feature_values,
        covariate_matrix,
    )

    partial_result = spearmanr(
        residual_feature,
        residual_rescue,
    )

    feature_rescue_rows.append({
        "feature": feature,

        "raw_spearman_rho": (
            raw_result.statistic
        ),
        "raw_p_value": (
            raw_result.pvalue
        ),

        "partial_spearman_rho": (
            partial_result.statistic
        ),
        "partial_p_value": (
            partial_result.pvalue
        ),

        "feature_mean": (
            np.mean(feature_values)
        ),
        "feature_std": (
            np.std(
                feature_values,
                ddof=1,
            )
        ),
    })


feature_rescue_tests = pd.DataFrame(
    feature_rescue_rows
)


# Correct separately for raw and partial tests.
feature_rescue_tests[
    "raw_fdr_q"
] = multipletests(
    feature_rescue_tests[
        "raw_p_value"
    ],
    method="fdr_bh",
)[1]

feature_rescue_tests[
    "partial_fdr_q"
] = multipletests(
    feature_rescue_tests[
        "partial_p_value"
    ],
    method="fdr_bh",
)[1]


feature_rescue_tests = (
    feature_rescue_tests
    .sort_values(
        [
            "partial_fdr_q",
            "partial_p_value",
        ]
    )
    .reset_index(drop=True)
)


print(
    "Feature associations with unrelated-promoter "
    "MLP rescue:"
)

display(feature_rescue_tests)


print(
    "\nFeatures significant before adjustment "
    "(raw FDR q < 0.05):"
)

display(
    feature_rescue_tests[
        feature_rescue_tests[
            "raw_fdr_q"
        ] < 0.05
    ]
)


print(
    "\nFeatures remaining significant after "
    "controlling donor probe strength and "
    "recipient variables (partial FDR q < 0.05):"
)

display(
    feature_rescue_tests[
        feature_rescue_tests[
            "partial_fdr_q"
        ] < 0.05
    ]
)


# Multivariable model to estimate combined explanatory value.
continuous_model_features = [
    "gc_fraction",
    "cpg_density",
    "cpg_observed_expected",
    "gpc_density",
    "sequence_entropy",
    "longest_homopolymer_run",
    "tata_min_hamming",
    "caat_min_hamming",
    "gc_box_min_hamming",
    "gc_box_exact_count",
    "initiator_match_count",
]

baseline_model_features = [
    "donor_probe_logit",
    "recipient_corrupt_logit",
    "recipient_gap",
]

full_model_features = (
    baseline_model_features
    + continuous_model_features
)


baseline_model = make_pipeline(
    StandardScaler(),
    LinearRegression(),
)

full_model = make_pipeline(
    StandardScaler(),
    LinearRegression(),
)


X_baseline = rescue_feature_df[
    baseline_model_features
].to_numpy(dtype=float)

X_full = rescue_feature_df[
    full_model_features
].to_numpy(dtype=float)

y = rescue_values


baseline_model.fit(
    X_baseline,
    y,
)

full_model.fit(
    X_full,
    y,
)


baseline_r2 = baseline_model.score(
    X_baseline,
    y,
)

full_r2 = full_model.score(
    X_full,
    y,
)


model_comparison_summary = pd.DataFrame([{
    "baseline_features": ", ".join(
        baseline_model_features
    ),
    "baseline_training_r2": baseline_r2,
    "full_training_r2": full_r2,
    "sequence_feature_incremental_r2": (
        full_r2 - baseline_r2
    ),
    "number_of_sequences": len(
        rescue_feature_df
    ),
}])


print("\nMultivariable model comparison:")
display(model_comparison_summary)


# Save results.
feature_rescue_tests.to_csv(
    CAUSAL_DIR
    / "notebook5_feature_rescue_associations.csv",
    index=False,
)

model_comparison_summary.to_csv(
    CAUSAL_DIR
    / "notebook5_feature_rescue_model_comparison.csv",
    index=False,
)

print(
    "\nSaved rescue-feature analyses to:",
    CAUSAL_DIR,
)

Feature associations with unrelated-promoter MLP rescue:


,feature,raw_spearman_rho,raw_p_value,partial_spearman_rho,partial_p_value,feature_mean,feature_std,raw_fdr_q,partial_fdr_q
0,gc_fraction,0.549551,2.632568e-16,0.452038,6.604563e-11,0.643764,0.128602,4.475366e-15,1.122776e-09
1,gpc_density,0.512694,4.629474e-14,0.412858,3.554429e-09,0.112721,0.054848,3.935053e-13,3.021264e-08
2,cpg_density,0.482784,1.988632e-12,0.373917,1.158520e-07,0.080209,0.060212,8.451688e-12,6.564945e-07
3,tata_min_hamming,0.409576,4.852758e-09,0.333319,2.776387e-06,2.492063,0.860598,1.649938e-08,1.179964e-05
4,sequence_entropy,-0.502191,1.807782e-13,-0.305071,1.968709e-05,1.834495,0.128212,1.024410e-12,6.693611e-05
5,cpg_observed_expected,0.377100,8.860771e-08,0.296472,3.438604e-05,0.673228,0.395597,2.510552e-07,9.742710e-05
6,gc_box_min_hamming,-0.338278,1.928168e-06,-0.275071,1.277946e-04,1.042328,0.742557,4.682693e-06,3.103583e-04
7,gc_box_exact_count,0.263188,2.532403e-04,0.144053,4.797434e-02,0.312169,0.629851,5.381356e-04,1.019455e-01
8,gc_box_present,0.254787,4.030488e-04,0.116632,1.099880e-01,0.238095,0.427049,7.613144e-04,2.077550e-01
9,initiator_match_count,-0.218423,2.532392e-03,-0.100592,1.684406e-01,0.269841,0.522057,4.305066e-03,2.760335e-01



Features significant before adjustment (raw FDR q < 0.05):


,feature,raw_spearman_rho,raw_p_value,partial_spearman_rho,partial_p_value,feature_mean,feature_std,raw_fdr_q,partial_fdr_q
0,gc_fraction,0.549551,2.632568e-16,0.452038,6.604563e-11,0.643764,0.128602,4.475366e-15,1.122776e-09
1,gpc_density,0.512694,4.629474e-14,0.412858,3.554429e-09,0.112721,0.054848,3.935053e-13,3.021264e-08
2,cpg_density,0.482784,1.988632e-12,0.373917,1.158520e-07,0.080209,0.060212,8.451688e-12,6.564945e-07
3,tata_min_hamming,0.409576,4.852758e-09,0.333319,2.776387e-06,2.492063,0.860598,1.649938e-08,1.179964e-05
4,sequence_entropy,-0.502191,1.807782e-13,-0.305071,1.968709e-05,1.834495,0.128212,1.024410e-12,6.693611e-05
5,cpg_observed_expected,0.377100,8.860771e-08,0.296472,3.438604e-05,0.673228,0.395597,2.510552e-07,9.742710e-05
6,gc_box_min_hamming,-0.338278,1.928168e-06,-0.275071,1.277946e-04,1.042328,0.742557,4.682693e-06,3.103583e-04
7,gc_box_exact_count,0.263188,2.532403e-04,0.144053,4.797434e-02,0.312169,0.629851,5.381356e-04,1.019455e-01
8,gc_box_present,0.254787,4.030488e-04,0.116632,1.099880e-01,0.238095,0.427049,7.613144e-04,2.077550e-01
9,initiator_match_count,-0.218423,2.532392e-03,-0.100592,1.684406e-01,0.269841,0.522057,4.305066e-03,2.760335e-01



Features remaining significant after controlling donor probe strength and recipient variables (partial FDR q < 0.05):


,feature,raw_spearman_rho,raw_p_value,partial_spearman_rho,partial_p_value,feature_mean,feature_std,raw_fdr_q,partial_fdr_q
0,gc_fraction,0.549551,2.632568e-16,0.452038,6.604563e-11,0.643764,0.128602,4.475366e-15,1.122776e-09
1,gpc_density,0.512694,4.629474e-14,0.412858,3.554429e-09,0.112721,0.054848,3.935053e-13,3.021264e-08
2,cpg_density,0.482784,1.988632e-12,0.373917,1.158520e-07,0.080209,0.060212,8.451688e-12,6.564945e-07
3,tata_min_hamming,0.409576,4.852758e-09,0.333319,2.776387e-06,2.492063,0.860598,1.649938e-08,1.179964e-05
4,sequence_entropy,-0.502191,1.807782e-13,-0.305071,1.968709e-05,1.834495,0.128212,1.024410e-12,6.693611e-05
5,cpg_observed_expected,0.377100,8.860771e-08,0.296472,3.438604e-05,0.673228,0.395597,2.510552e-07,9.742710e-05
6,gc_box_min_hamming,-0.338278,1.928168e-06,-0.275071,1.277946e-04,1.042328,0.742557,4.682693e-06,3.103583e-04



Multivariable model comparison:


,baseline_features,baseline_training_r2,full_training_r2,sequence_feature_incremental_r2,number_of_sequences
0,"donor_probe_logit, recipient_corrupt_logit, re...",0.416491,0.545218,0.128727,189



Saved rescue-feature analyses to: /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_causal_interventions


# Cell 8 — Cross-validated test of whether biological features
# predict rescue beyond donor probe strength and recipient difficulty

In [8]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

CV_SEED = 42
N_SPLITS = 10

baseline_feature_names = [
    "donor_probe_logit",
    "recipient_corrupt_logit",
    "recipient_gap",
]

# Use a reduced, biologically interpretable feature set.
# Avoid including many near-duplicate motif encodings.
sequence_feature_names = [
    "gc_fraction",
    "cpg_observed_expected",
    "sequence_entropy",
    "gc_box_min_hamming",
    "tata_min_hamming",
    "longest_homopolymer_run",
]

full_feature_names = (
    baseline_feature_names
    + sequence_feature_names
)

X_baseline = rescue_feature_df[
    baseline_feature_names
].to_numpy(dtype=float)

X_full = rescue_feature_df[
    full_feature_names
].to_numpy(dtype=float)

y = rescue_feature_df[
    "unrelated_rescue_amount"
].to_numpy(dtype=float)


cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CV_SEED,
)

baseline_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0),
)

full_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0),
)


baseline_cv_r2 = cross_val_score(
    baseline_model,
    X_baseline,
    y,
    cv=cv,
    scoring="r2",
)

full_cv_r2 = cross_val_score(
    full_model,
    X_full,
    y,
    cv=cv,
    scoring="r2",
)

paired_cv_improvement = (
    full_cv_r2 - baseline_cv_r2
)


cv_model_summary = pd.DataFrame([
    {
        "model": "baseline covariates",
        "number_of_features": len(
            baseline_feature_names
        ),
        "mean_cv_r2": (
            baseline_cv_r2.mean()
        ),
        "std_cv_r2": (
            baseline_cv_r2.std(ddof=1)
        ),
        "median_cv_r2": np.median(
            baseline_cv_r2
        ),
    },
    {
        "model": (
            "baseline plus biological features"
        ),
        "number_of_features": len(
            full_feature_names
        ),
        "mean_cv_r2": (
            full_cv_r2.mean()
        ),
        "std_cv_r2": (
            full_cv_r2.std(ddof=1)
        ),
        "median_cv_r2": np.median(
            full_cv_r2
        ),
    },
])

print("Cross-validated model performance:")
display(cv_model_summary)


cv_improvement_summary = pd.DataFrame([{
    "number_of_folds": N_SPLITS,
    "mean_incremental_cv_r2": (
        paired_cv_improvement.mean()
    ),
    "median_incremental_cv_r2": (
        np.median(paired_cv_improvement)
    ),
    "minimum_incremental_cv_r2": (
        paired_cv_improvement.min()
    ),
    "maximum_incremental_cv_r2": (
        paired_cv_improvement.max()
    ),
    "fraction_folds_improved": np.mean(
        paired_cv_improvement > 0
    ),
}])

print(
    "\nPaired improvement from adding biological features:"
)
display(cv_improvement_summary)


fold_results_df = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),
    "baseline_cv_r2": baseline_cv_r2,
    "full_cv_r2": full_cv_r2,
    "incremental_cv_r2": (
        paired_cv_improvement
    ),
})

print("\nPer-fold results:")
display(fold_results_df)


# Obtain out-of-fold predictions for a direct overall R² comparison.
baseline_oof_predictions = np.empty_like(
    y,
    dtype=float,
)

full_oof_predictions = np.empty_like(
    y,
    dtype=float,
)

for train_indices, test_indices in cv.split(
    X_full
):
    fold_baseline_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0),
    )

    fold_full_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0),
    )

    fold_baseline_model.fit(
        X_baseline[train_indices],
        y[train_indices],
    )

    fold_full_model.fit(
        X_full[train_indices],
        y[train_indices],
    )

    baseline_oof_predictions[
        test_indices
    ] = fold_baseline_model.predict(
        X_baseline[test_indices]
    )

    full_oof_predictions[
        test_indices
    ] = fold_full_model.predict(
        X_full[test_indices]
    )


overall_oof_summary = pd.DataFrame([{
    "baseline_oof_r2": r2_score(
        y,
        baseline_oof_predictions,
    ),
    "full_oof_r2": r2_score(
        y,
        full_oof_predictions,
    ),
    "incremental_oof_r2": (
        r2_score(
            y,
            full_oof_predictions,
        )
        - r2_score(
            y,
            baseline_oof_predictions,
        )
    ),
    "baseline_oof_correlation": (
        np.corrcoef(
            y,
            baseline_oof_predictions,
        )[0, 1]
    ),
    "full_oof_correlation": (
        np.corrcoef(
            y,
            full_oof_predictions,
        )[0, 1]
    ),
}])

print("\nOverall out-of-fold performance:")
display(overall_oof_summary)


# Fit the full model on all observations only to inspect standardized
# coefficient directions. These coefficients are descriptive, not causal.
full_model.fit(
    X_full,
    y,
)

ridge_model = full_model.named_steps[
    "ridge"
]

standardized_coefficient_df = pd.DataFrame({
    "feature": full_feature_names,
    "standardized_coefficient": (
        ridge_model.coef_
    ),
}).sort_values(
    "standardized_coefficient",
    key=np.abs,
    ascending=False,
)

print(
    "\nStandardized Ridge coefficients:"
)
display(standardized_coefficient_df)


# Save all results.
cv_model_summary.to_csv(
    CAUSAL_DIR
    / "notebook5_cross_validated_model_summary.csv",
    index=False,
)

cv_improvement_summary.to_csv(
    CAUSAL_DIR
    / "notebook5_cross_validated_incremental_r2.csv",
    index=False,
)

fold_results_df.to_csv(
    CAUSAL_DIR
    / "notebook5_cross_validation_fold_results.csv",
    index=False,
)

overall_oof_summary.to_csv(
    CAUSAL_DIR
    / "notebook5_out_of_fold_model_summary.csv",
    index=False,
)

standardized_coefficient_df.to_csv(
    CAUSAL_DIR
    / "notebook5_ridge_standardized_coefficients.csv",
    index=False,
)

print(
    "\nSaved cross-validated analyses to:",
    CAUSAL_DIR,
)

Cross-validated model performance:


,model,number_of_features,mean_cv_r2,std_cv_r2,median_cv_r2
0,baseline covariates,3,0.271794,0.360916,0.293697
1,baseline plus biological features,9,0.327355,0.512025,0.433547



Paired improvement from adding biological features:


,number_of_folds,mean_incremental_cv_r2,median_incremental_cv_r2,minimum_incremental_cv_r2,maximum_incremental_cv_r2,fraction_folds_improved
0,10,0.05556,0.114424,-0.343326,0.240095,0.7



Per-fold results:


...


Overall out-of-fold performance:


,baseline_oof_r2,full_oof_r2,incremental_oof_r2,baseline_oof_correlation,full_oof_correlation
0,0.381645,0.471566,0.089921,0.61864,0.687742



Standardized Ridge coefficients:


,feature,standardized_coefficient
1,recipient_corrupt_logit,-4.517703
0,donor_probe_logit,3.626968
3,gc_fraction,2.872279
7,tata_min_hamming,1.064439
4,cpg_observed_expected,0.725043
2,recipient_gap,-0.597057
8,longest_homopolymer_run,-0.310378
6,gc_box_min_hamming,0.191122
5,sequence_entropy,-0.170315



Saved cross-validated analyses to: /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_causal_interventions


# Cell 9 — Repeated cross-validation:
# baseline versus GC-only versus full biological feature models

In [9]:
from sklearn.model_selection import RepeatedKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

REPEATED_CV_SEED = 42
N_SPLITS = 5
N_REPEATS = 50

baseline_feature_names = [
    "donor_probe_logit",
    "recipient_corrupt_logit",
    "recipient_gap",
]

gc_model_feature_names = (
    baseline_feature_names
    + ["gc_fraction"]
)

full_biological_feature_names = (
    baseline_feature_names
    + [
        "gc_fraction",
        "cpg_observed_expected",
        "sequence_entropy",
        "gc_box_min_hamming",
        "tata_min_hamming",
        "longest_homopolymer_run",
    ]
)

X_baseline = rescue_feature_df[
    baseline_feature_names
].to_numpy(dtype=float)

X_gc = rescue_feature_df[
    gc_model_feature_names
].to_numpy(dtype=float)

X_full = rescue_feature_df[
    full_biological_feature_names
].to_numpy(dtype=float)

y = rescue_feature_df[
    "unrelated_rescue_amount"
].to_numpy(dtype=float)


repeated_cv = RepeatedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=REPEATED_CV_SEED,
)


repeated_cv_rows = []

for split_index, (
    train_indices,
    test_indices,
) in enumerate(
    repeated_cv.split(X_full)
):
    repeat_index = (
        split_index // N_SPLITS
    )

    fold_index = (
        split_index % N_SPLITS
    )

    models_and_data = {
        "baseline": X_baseline,
        "baseline_plus_gc": X_gc,
        "baseline_plus_full_biology": X_full,
    }

    fold_scores = {}

    for model_name, feature_matrix in (
        models_and_data.items()
    ):
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=1.0),
        )

        model.fit(
            feature_matrix[train_indices],
            y[train_indices],
        )

        predictions = model.predict(
            feature_matrix[test_indices]
        )

        fold_scores[model_name] = r2_score(
            y[test_indices],
            predictions,
        )

    repeated_cv_rows.append({
        "repeat": repeat_index,
        "fold": fold_index,
        "baseline_r2": (
            fold_scores["baseline"]
        ),
        "gc_model_r2": (
            fold_scores[
                "baseline_plus_gc"
            ]
        ),
        "full_biology_r2": (
            fold_scores[
                "baseline_plus_full_biology"
            ]
        ),
        "gc_incremental_r2": (
            fold_scores[
                "baseline_plus_gc"
            ]
            - fold_scores["baseline"]
        ),
        "full_incremental_r2": (
            fold_scores[
                "baseline_plus_full_biology"
            ]
            - fold_scores["baseline"]
        ),
        "additional_biology_beyond_gc_r2": (
            fold_scores[
                "baseline_plus_full_biology"
            ]
            - fold_scores[
                "baseline_plus_gc"
            ]
        ),
    })


repeated_cv_df = pd.DataFrame(
    repeated_cv_rows
)

print(
    "Repeated CV fold-level results:",
    len(repeated_cv_df),
)

display(repeated_cv_df.head(10))


repeated_cv_summary = pd.DataFrame([
    {
        "comparison": "GC added to baseline",
        "mean_incremental_r2": (
            repeated_cv_df[
                "gc_incremental_r2"
            ].mean()
        ),
        "median_incremental_r2": (
            repeated_cv_df[
                "gc_incremental_r2"
            ].median()
        ),
        "std_incremental_r2": (
            repeated_cv_df[
                "gc_incremental_r2"
            ].std(ddof=1)
        ),
        "fraction_folds_improved": (
            repeated_cv_df[
                "gc_incremental_r2"
            ] > 0
        ).mean(),
    },
    {
        "comparison": (
            "Full biology added to baseline"
        ),
        "mean_incremental_r2": (
            repeated_cv_df[
                "full_incremental_r2"
            ].mean()
        ),
        "median_incremental_r2": (
            repeated_cv_df[
                "full_incremental_r2"
            ].median()
        ),
        "std_incremental_r2": (
            repeated_cv_df[
                "full_incremental_r2"
            ].std(ddof=1)
        ),
        "fraction_folds_improved": (
            repeated_cv_df[
                "full_incremental_r2"
            ] > 0
        ).mean(),
    },
    {
        "comparison": (
            "Additional biology beyond GC"
        ),
        "mean_incremental_r2": (
            repeated_cv_df[
                "additional_biology_beyond_gc_r2"
            ].mean()
        ),
        "median_incremental_r2": (
            repeated_cv_df[
                "additional_biology_beyond_gc_r2"
            ].median()
        ),
        "std_incremental_r2": (
            repeated_cv_df[
                "additional_biology_beyond_gc_r2"
            ].std(ddof=1)
        ),
        "fraction_folds_improved": (
            repeated_cv_df[
                "additional_biology_beyond_gc_r2"
            ] > 0
        ).mean(),
    },
])

print("\nRepeated cross-validation summary:")
display(repeated_cv_summary)


# Aggregate within each repeat so the five folds from the
# same split are not treated as independent observations.
repeat_level_summary = (
    repeated_cv_df
    .groupby("repeat")
    .agg(
        baseline_mean_r2=(
            "baseline_r2",
            "mean",
        ),
        gc_model_mean_r2=(
            "gc_model_r2",
            "mean",
        ),
        full_biology_mean_r2=(
            "full_biology_r2",
            "mean",
        ),
        gc_incremental_mean_r2=(
            "gc_incremental_r2",
            "mean",
        ),
        full_incremental_mean_r2=(
            "full_incremental_r2",
            "mean",
        ),
        additional_biology_beyond_gc_mean_r2=(
            "additional_biology_beyond_gc_r2",
            "mean",
        ),
    )
    .reset_index()
)

print("\nRepeat-level summary:")
display(repeat_level_summary.head(10))


# Bootstrap across the 50 repeat-level averages.
N_BOOTSTRAPS = 5000
bootstrap_rng = np.random.default_rng(42)

bootstrap_rows = []

for column, label in [
    (
        "gc_incremental_mean_r2",
        "GC added to baseline",
    ),
    (
        "full_incremental_mean_r2",
        "Full biology added to baseline",
    ),
    (
        "additional_biology_beyond_gc_mean_r2",
        "Additional biology beyond GC",
    ),
]:
    values = repeat_level_summary[
        column
    ].to_numpy()

    bootstrap_means = np.empty(
        N_BOOTSTRAPS
    )

    for bootstrap_index in range(
        N_BOOTSTRAPS
    ):
        sampled_indices = (
            bootstrap_rng.integers(
                0,
                len(values),
                size=len(values),
            )
        )

        bootstrap_means[
            bootstrap_index
        ] = values[
            sampled_indices
        ].mean()

    bootstrap_rows.append({
        "comparison": label,
        "mean_incremental_r2": (
            values.mean()
        ),
        "ci_2.5": np.percentile(
            bootstrap_means,
            2.5,
        ),
        "ci_97.5": np.percentile(
            bootstrap_means,
            97.5,
        ),
        "fraction_repeats_positive": (
            values > 0
        ).mean(),
    })


repeated_cv_bootstrap_summary = (
    pd.DataFrame(bootstrap_rows)
)

print(
    "\nBootstrap summary across repeated CV splits:"
)

display(repeated_cv_bootstrap_summary)


repeated_cv_df.to_csv(
    CAUSAL_DIR
    / "notebook5_repeated_cv_fold_results.csv",
    index=False,
)

repeat_level_summary.to_csv(
    CAUSAL_DIR
    / "notebook5_repeated_cv_repeat_summary.csv",
    index=False,
)

repeated_cv_bootstrap_summary.to_csv(
    CAUSAL_DIR
    / "notebook5_repeated_cv_bootstrap_summary.csv",
    index=False,
)

print(
    "\nSaved repeated cross-validation results to:",
    CAUSAL_DIR,
)

Repeated CV fold-level results: 250


,repeat,fold,baseline_r2,gc_model_r2,full_biology_r2,gc_incremental_r2,full_incremental_r2,additional_biology_beyond_gc_r2
0,0,0,0.273761,0.385712,0.388908,0.111950,0.115147,0.003196
1,0,1,0.411864,0.450633,0.419693,0.038769,0.007829,-0.030940
2,0,2,0.320910,0.470753,0.446885,0.149842,0.125975,-0.023868
3,0,3,0.394373,0.376964,0.355774,-0.017409,-0.038599,-0.021189
4,0,4,0.461241,0.628606,0.621794,0.167364,0.160552,-0.006812
5,1,0,0.463800,0.562977,0.456659,0.099177,-0.007142,-0.106318
6,1,1,0.193806,0.160634,0.142550,-0.033172,-0.051257,-0.018085
7,1,2,0.316233,0.399439,0.393010,0.083207,0.076777,-0.006429
8,1,3,0.437954,0.573636,0.472275,0.135682,0.034321,-0.101361
9,1,4,0.324891,0.510339,0.504023,0.185449,0.179132,-0.006316



Repeated cross-validation summary:


,comparison,mean_incremental_r2,median_incremental_r2,std_incremental_r2,fraction_folds_improved
0,GC added to baseline,0.105567,0.111567,0.071236,0.920
1,Full biology added to baseline,0.072174,0.083272,0.078946,0.816
2,Additional biology beyond GC,-0.033393,-0.027185,0.033279,0.128



Repeat-level summary:


,repeat,baseline_mean_r2,gc_model_mean_r2,full_biology_mean_r2,gc_incremental_mean_r2,full_incremental_mean_r2,additional_biology_beyond_gc_mean_r2
0,0,0.372430,0.462533,0.446611,0.090103,0.074181,-0.015923
1,1,0.347337,0.441405,0.393703,0.094068,0.046366,-0.047702
2,2,0.314421,0.410321,0.366017,0.095901,0.051597,-0.044304
3,3,0.378718,0.488035,0.439698,0.109317,0.060980,-0.048338
4,4,0.352459,0.461800,0.448797,0.109341,0.096338,-0.013003
5,5,0.330086,0.445525,0.414677,0.115440,0.084591,-0.030849
6,6,0.313969,0.428082,0.385432,0.114113,0.071463,-0.042650
7,7,0.374825,0.477945,0.429423,0.103120,0.054599,-0.048521
8,8,0.346723,0.442916,0.391614,0.096193,0.044891,-0.051302
9,9,0.379490,0.489590,0.436011,0.110100,0.056521,-0.053579



Bootstrap summary across repeated CV splits:


,comparison,mean_incremental_r2,ci_2.5,ci_97.5,fraction_repeats_positive
0,GC added to baseline,0.105567,0.101730,0.109320,1.0
1,Full biology added to baseline,0.072174,0.067566,0.076961,1.0
2,Additional biology beyond GC,-0.033393,-0.037020,-0.029697,0.0



Saved repeated cross-validation results to: /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_causal_interventions


# Cell 10 — Final Notebook 5 summary and conclusions

In [10]:
notebook5_result_summary = pd.DataFrame([
    {
        "finding": (
            "Promoter cohort GC enrichment"
        ),
        "result": (
            "Promoter mean GC = 0.644; "
            "non-promoter mean GC = 0.543"
        ),
        "interpretation": (
            "The probe-recognized rescue cohort is "
            "strongly enriched for GC-rich promoters."
        ),
    },
    {
        "finding": (
            "GC-box enrichment"
        ),
        "result": (
            "GC-box present in 23.8% of promoters "
            "versus 8.5% of non-promoters"
        ),
        "interpretation": (
            "GC-box-like motifs are enriched, but are "
            "not independently predictive beyond GC."
        ),
    },
    {
        "finding": (
            "GC-rescue association"
        ),
        "result": (
            "Raw Spearman rho = 0.550; "
            "adjusted rho = 0.452"
        ),
        "interpretation": (
            "Higher-GC promoter donors produce stronger "
            "Layer-1 MLP transfer effects."
        ),
    },
    {
        "finding": (
            "GC cross-validated contribution"
        ),
        "result": (
            "Mean incremental repeated-CV R2 = 0.106; "
            "92% of folds improved"
        ),
        "interpretation": (
            "GC adds reproducible predictive information "
            "beyond donor probe score and recipient difficulty."
        ),
    },
    {
        "finding": (
            "Additional biology beyond GC"
        ),
        "result": (
            "Mean incremental repeated-CV R2 = -0.033"
        ),
        "interpretation": (
            "The tested motifs and composition features do "
            "not improve generalization after GC is included."
        ),
    },
])

display(notebook5_result_summary)


notebook5_conclusions = """
# Notebook 5: Biological Interpretation of the Layer-1 MLP Signal

## Objective

Determine which sequence properties are associated with the
transferable promoter-class representation found in the
Layer-1 MLP of baseline DNABERT-2.

## Dataset

The analysis used:

- 189 probe-recognized promoter sequences
- 189 unrelated-promoter donor interventions
- 189 sampled non-promoter sequences

The promoter cohort was selected because:

1. the fixed Layer-5 probe classified the sequence as promoter-like;
2. exact dinucleotide-preserving corruption reduced the probe
   logit by at least one unit.

Therefore, the cohort is not a representative sample of all
promoter architectures.

## Promoter versus non-promoter sequence features

The selected promoters were substantially more GC-rich:

- promoter mean GC fraction: 0.644
- non-promoter mean GC fraction: 0.543

Promoters also had:

- lower nucleotide entropy;
- higher GpC density;
- modestly higher CpG density;
- greater enrichment for GC-box-like motifs;
- fewer TATA-like sequences.

GC-box-like motifs occurred in:

- 23.8% of promoters
- 8.5% of non-promoters

The odds ratio was approximately 3.38.

These results describe the selected probe-recognized cohort and
should not be generalized to all promoters.

## Features associated with MLP transfer

Among unrelated promoter donors, stronger Layer-1 MLP rescue was
associated with:

- higher GC fraction;
- higher GpC density;
- higher CpG density;
- lower sequence entropy;
- greater distance from a TATA-like sequence;
- closer similarity to a GC-box-like motif.

GC fraction was the strongest individual predictor:

- raw Spearman rho: 0.550
- adjusted Spearman rho: 0.452

The adjusted analysis accounted for:

- donor Layer-5 probe logit;
- recipient corrupted logit;
- recipient clean-corrupt gap.

## Cross-validated predictive value

A baseline model using donor probe strength and recipient
difficulty achieved an out-of-fold R2 of approximately 0.382.

Adding biological sequence features increased out-of-fold R2 to
approximately 0.472.

Repeated cross-validation showed that GC content alone was the
most reliable addition:

- mean incremental R2: 0.106
- median incremental R2: 0.112
- fraction of folds improved: 0.920
- fraction of repeated splits with positive mean improvement: 1.0

Adding the full biological feature panel produced a smaller
mean improvement:

- mean incremental R2: 0.072

Features beyond GC reduced generalization:

- mean incremental R2 beyond GC: -0.033
- fraction of folds improved: 0.128
- fraction of repeated splits positive: 0.0

## Supported interpretation

GC content is the dominant simple sequence property associated
with the transferable Layer-1 MLP promoter representation.

Higher-GC promoter donors transfer stronger promoter-associated
evidence into corrupted recipients, even after accounting for
the donor's final Layer-5 probe score and recipient difficulty.

CpG density, GpC density, sequence entropy, and GC-box similarity
are correlated with the transfer effect, but they do not provide
reliable predictive value beyond total GC content.

## Interpretation boundary

The current evidence does not establish that:

- the Layer-1 MLP explicitly calculates GC percentage;
- GC content is the only feature encoded;
- the representation is specific to biological promoter function;
- GC content causally generates the Layer-1 MLP signal;
- the result generalizes to all promoter subclasses.

GC may be:

1. the directly encoded variable;
2. a proxy for GC-rich promoter architecture;
3. a proxy for CpG-island-like sequence structure;
4. correlated with a more complex learned sequence feature.

## Next experiment

Notebook 6 will causally manipulate GC content while controlling
sequence length and, where possible, dinucleotide or motif
structure.

The primary test is:

Does deliberately increasing or decreasing GC content produce a
corresponding change in:

1. Layer-1 MLP activation;
2. Layer-5 promoter probe evidence;
3. transferable MLP rescue strength?

Controlled interventions are required before making a mechanistic
claim about GC encoding.
"""


NOTEBOOK5_DIR = (
    OUTPUT_DIR
    / "promoter_mlp_biological_interpretation"
)

NOTEBOOK5_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

notebook5_result_summary.to_csv(
    NOTEBOOK5_DIR
    / "notebook5_result_summary.csv",
    index=False,
)

with open(
    NOTEBOOK5_DIR
    / "NOTEBOOK_5_CONCLUSIONS.md",
    "w",
) as file:
    file.write(
        notebook5_conclusions.strip()
    )

print("\nSaved Notebook 5 summary to:")
print(
    NOTEBOOK5_DIR
    / "notebook5_result_summary.csv"
)

print("\nSaved Notebook 5 conclusions to:")
print(
    NOTEBOOK5_DIR
    / "NOTEBOOK_5_CONCLUSIONS.md"
)

print("\nNotebook 5 complete.")

,finding,result,interpretation
0,Promoter cohort GC enrichment,Promoter mean GC = 0.644; non-promoter mean GC...,The probe-recognized rescue cohort is strongly...
1,GC-box enrichment,GC-box present in 23.8% of promoters versus 8....,"GC-box-like motifs are enriched, but are not i..."
2,GC-rescue association,Raw Spearman rho = 0.550; adjusted rho = 0.452,Higher-GC promoter donors produce stronger Lay...
3,GC cross-validated contribution,Mean incremental repeated-CV R2 = 0.106; 92% o...,GC adds reproducible predictive information be...
4,Additional biology beyond GC,Mean incremental repeated-CV R2 = -0.033,The tested motifs and composition features do ...



Saved Notebook 5 summary to:
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_biological_interpretation/notebook5_result_summary.csv

Saved Notebook 5 conclusions to:
/content/drive/MyDrive/algoverse/dnabert/outputs/promoter_mlp_biological_interpretation/NOTEBOOK_5_CONCLUSIONS.md

Notebook 5 complete.
